## Free Throw Model — Cross‑Session Evaluation & Calibration

### Prerequisites: 
* Run feature engineering scripts (extract_phase_features.py and prepare_phase_dataset.py) to get features. 

### Machine Learning Pipeline:
- #### **Step 1** Split data between training/validation and testing
    - Right now I’m using 80% of the data for training/validation and 20% for testing.
- #### **Step 2** Hyperparemeter Tuning 
    - k fold cross validation and a grid search to find the hyperparameters for each model that optimize f1 score
- #### **Step 3** Calibration/Thresholding on best CV models
    - two **calibration** methods:
        - sigmoid
        - isotonic 
    - two **thresholding** methods: 
        - highest F1 score possible
        - F1 score measured at the threshold where precision is constrained to 70%
- #### **Step 4** Final Training and Testing
    - train best performing models on all training data
    - plot confusion matrices, reliability curves, and precision recall curves 
- #### **Step 5** Diagnostics of Final Models
    - Test the model on the 20% testing data and see how it does on data that it has not seen before
- #### **Step 6** Feature Pruning
    - From permutation Importance

# TODO 
* Better Analysis/Interpretation
* Better Model Performance: (PR Curves, ROC curves, calibration curves)
* Feature Importance: coefficients (LogReg), impurity (RF/GB), SHAP/Permutation importances
* Error analysis: confusion matrix, misclassified samples.
* Save everything: model weights, plots, tables, config
* Make yaml config files control all configuration in this script 

In [134]:
# =================================================
# Create unique experiment directory 
# =================================================

from datetime import datetime
import re
from pathlib import Path
import yaml

PROJECT_ROOT = Path().resolve().parent # go up one level

# Load feature config
feature_cfg_path = PROJECT_ROOT / "feature_config.yaml"
with open(feature_cfg_path, "r") as f:
    feature_config = yaml.safe_load(f)

FEATURE_VERSION = feature_config["feature_version"]
MODEL_TYPE = feature_config["model_type"]   # "summary_stats" or "time_series"

# --- makes label for the experiment ---
# def start_run(label: str = "baseline") -> Path:
#     ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
#     root = PROJECT_ROOT / "experiments"
#     root.mkdir(parents=True, exist_ok=True)
#     # auto-increment run index if we collide on timestamp+label
#     existing = sorted(root.glob(f"{ts}_run-*_{label}"))
#     idx = 1
#     if existing:
#         m = re.search(r"_run-(\d{3})_", existing[-1].name)
#         idx = (int(m.group(1)) + 1) if m else 1
#     rd = root / f"{ts}_run-{idx:03d}_{label}"
#     (rd ).mkdir(parents=True, exist_ok=True)
#     print(f"[INFO] Run directory: {rd}")
#     return rd

def start_run(label: str = "baseline") -> Path:
    ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    
    # put MODEL_TYPE under experiments
    root = PROJECT_ROOT / "experiments" / MODEL_TYPE
    root.mkdir(parents=True, exist_ok=True)
    
    existing = sorted(root.glob(f"{ts}_run-*_{label}"))
    idx = 1
    if existing:
        m = re.search(r"_run-(\d{3})_", existing[-1].name)
        idx = (int(m.group(1)) + 1) if m else 1
    
    rd = root / f"{ts}_run-{idx:03d}_{label}"
    rd.mkdir(parents=True, exist_ok=True)
    print(f"[INFO] Run directory: {rd}")
    return rd

run_dir = start_run("baseline")

[INFO] Run directory: /Users/kwill55/RVL/personalized-sports-optimization/experiments/summary_stats/2025-08-21_20-57-18_run-001_baseline


In [135]:
# =================================================
# Step 0: Import Libraries and Create Models
# =================================================

from __future__ import annotations
from pathlib import Path
from joblib import dump, load

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import json
import re

# --- sklearn ---
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score as _f1_for_perm
from sklearn.model_selection import GridSearchCV, StratifiedKFold, GroupKFold
from sklearn.metrics import (
    precision_recall_fscore_support, average_precision_score,
    roc_auc_score, brier_score_loss
)
from sklearn.metrics import (
    precision_score, recall_score, f1_score, 
    average_precision_score, roc_auc_score
)
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score, brier_score_loss,
    confusion_matrix
)

# --- Plotting Util (saves CM, PR, calibration) ---
from plots import save_all_eval_figures

from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score,
    brier_score_loss, confusion_matrix
)

# --- Load Project Config ---
project_cfg_path = PROJECT_ROOT / "project_config.yaml"
with open(project_cfg_path, "r") as f:
    project_config = yaml.safe_load(f)

ATHLETE = project_config["athlete"]
SESSION = project_config["session"]


# --- Load Feature Config ---
feature_cfg_path = PROJECT_ROOT / "feature_config.yaml"
with open(feature_cfg_path, "r") as f:
    feature_config = yaml.safe_load(f)

FEATURE_VERSION = feature_config["feature_version"] # version of feature data set 
MODEL_TYPE = feature_config["model_type"] # summary_stats or time_series

# --- Paths and Directories ---
DATA_ROOT = PROJECT_ROOT / "data" / ATHLETE

SESSIONS = {
    "session_02": DATA_ROOT / "session_02" / "analysis" / "datasets" / "features" / MODEL_TYPE / FEATURE_VERSION,
    "session_03": DATA_ROOT / "session_03" / "analysis" / "datasets" / "features" / MODEL_TYPE / FEATURE_VERSION,
    # add more sessions here...
}

MODELS = {
    "LogisticRegression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced"   # weight classes inversely to freq
    ),
    "LinearSVC": LinearSVC(
        C=1.0,
        max_iter=10000,
        tol=1e-3,
        class_weight="balanced"
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced"   # balances splits during tree construction
    ),
    "GradientBoosting": GradientBoostingClassifier(
        random_state=42
        # sklearn's GradientBoostingClassifier does NOT have class_weight
    ),
    "GaussianNB": GaussianNB()
    # Naive Bayes has no native class_weight 
}



In [137]:
# =================================================
# Step 1: Split data into train/validation and test
# =================================================

TEST_SIZE = 0.2
RANDOM_STATE = 42

X_train_list, X_test_list = [], []
y_train_list, y_test_list = [], []
session_train_list, session_test_list = [], []

for session_name, base in SESSIONS.items():
    # Load
    X = pd.read_csv(base / "X.csv")
    y = pd.read_csv(base / "y.csv").squeeze("columns").astype(int)

    # Stratify class proportions based on y (makes/misses)
    strat = y if y.nunique() > 1 else None
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=strat 
    )

    # Store with session labels
    X_train_list.append(X_tr) 
    X_test_list.append(X_te)
    y_train_list.append(y_tr)
    y_test_list.append(y_te)
    session_train_list.extend([session_name] * len(y_tr))
    session_test_list.extend([session_name] * len(y_te))

# Concatenate everything
X_train = pd.concat(X_train_list, ignore_index=True)
X_test  = pd.concat(X_test_list, ignore_index=True)
y_train = pd.concat(y_train_list, ignore_index=True)
y_test  = pd.concat(y_test_list, ignore_index=True)
session_train = pd.Series(session_train_list, name="session")
session_test  = pd.Series(session_test_list, name="session")

print("[INFO] Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("\n[TRAIN counts]")
print(pd.crosstab(session_train, y_train))
print("\n[TEST counts]")
print(pd.crosstab(session_test, y_test))


[INFO] Train shape: (224, 51) Test shape: (57, 51)

[TRAIN counts]
_label       0   1
session           
session_02  27  52
session_03  53  92

[TEST counts]
_label       0   1
session           
session_02   7  13
session_03  14  23


In [138]:
# =================================================
# Step 2: Tune model hyperparameters
# =================================================

try:
    groups = session_train   
except NameError:
    groups = None

# Wrap models that need scaling; leave tree/nb models raw
pipelines = {
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", MODELS["LogisticRegression"])
    ]),
    "LinearSVC": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", MODELS["LinearSVC"])
    ]),
    "RandomForest": Pipeline([
        ("clf", MODELS["RandomForest"])
    ]),
    "GradientBoosting": Pipeline([
        ("clf", MODELS["GradientBoosting"])
    ]),
    "GaussianNB": Pipeline([
        ("scaler", StandardScaler()),  
        ("clf", MODELS["GaussianNB"])
    ]),
}

# Hyperparameter grids for each model
param_grids = {
    "LogisticRegression": {
        "clf__C": [0.1, 0.3, 1.0, 3.0, 10.0],
        "clf__penalty": ["l2"],
        "clf__solver": ["lbfgs"],
    },
    "LinearSVC": {
        "clf__C": [0.1, 0.3, 1.0, 3.0, 10.0],
        "clf__tol": [1e-3, 1e-4],
    },
    "RandomForest": {
        "clf__n_estimators": [200, 300, 500],
        "clf__max_depth": [None, 8, 16],
        "clf__min_samples_split": [2, 5],
        "clf__max_features": ["sqrt", "log2", None],
    },
    "GradientBoosting": {
        "clf__n_estimators": [100, 200, 300],
        "clf__learning_rate": [0.03, 0.1, 0.3],
        "clf__max_depth": [2, 3, 4],
        "clf__subsample": [0.7, 1.0],
    },
    "GaussianNB": {
        "clf__var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6],
    },
}

# Cross Validation splitter 
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# TODO - add GroupKFold option later down the road 

scoring = {
    "f1": "f1",
    "ap": "average_precision",
    "roc": "roc_auc"
}

leaderboard_rows = []
best_models = {}

for name, pipe in pipelines.items():
    print(f"\n=== Tuning {name} ===")
    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grids[name],
        cv=cv,                    
        scoring={"f1": "f1", "ap": "average_precision", "roc": "roc_auc"},
        refit="f1",
        n_jobs=-1,
        verbose=0,
        return_train_score=False
    )
    grid.fit(X_train, y_train)

    best_models[name] = grid.best_estimator_

    row = {
        "model": name,
        "best_params": grid.best_params_,
        "cv_f1": grid.cv_results_["mean_test_f1"][grid.best_index_],
        "cv_ap": grid.cv_results_["mean_test_ap"][grid.best_index_],
        "cv_roc": grid.cv_results_["mean_test_roc"][grid.best_index_],
    }
    leaderboard_rows.append(row)

leaderboard = pd.DataFrame(leaderboard_rows).sort_values("cv_f1", ascending=False).reset_index(drop=True)
print("\n[LEADERBOARD — sorted by CV F1]")
display(leaderboard)

# Persist best models for this run
from joblib import dump
save_dir = run_dir / "cv_models"
save_dir.mkdir(parents=True, exist_ok=True)
for name, model in best_models.items():
    dump(model, save_dir / f"best_{name}.joblib")
print(f"[INFO] Saved tuned models to: {save_dir}")



=== Tuning LogisticRegression ===

=== Tuning LinearSVC ===

=== Tuning RandomForest ===

=== Tuning GradientBoosting ===

=== Tuning GaussianNB ===

[LEADERBOARD — sorted by CV F1]


,model,best_params,cv_f1,cv_ap,cv_roc
0,RandomForest,"{'clf__max_depth': 16, 'clf__max_features': 's...",0.776267,0.762158,0.634775
1,GradientBoosting,"{'clf__learning_rate': 0.03, 'clf__max_depth':...",0.767826,0.758640,0.635822
2,GaussianNB,{'clf__var_smoothing': 1e-09},0.752327,0.743437,0.608020
3,LogisticRegression,"{'clf__C': 3.0, 'clf__penalty': 'l2', 'clf__so...",0.725724,0.779566,0.691533
4,LinearSVC,"{'clf__C': 10.0, 'clf__tol': 0.001}",0.721917,0.790027,0.705973


[INFO] Saved tuned models to: /Users/kwill55/RVL/personalized-sports-optimization/experiments/summary_stats/2025-08-21_20-57-18_run-001_baseline/cv_models


In [139]:
# =================================================
# Step 3: Calibration & Threshold Tuning
# =================================================

# Split training data into training/validation
X_subtr, X_val, y_subtr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

def pick_threshold(p, y, precision_floor=None, tmin=0.05, tmax=0.95, n=181):
    ts = np.linspace(tmin, tmax, n)
    best = {"t": 0.5, "precision": 0.0, "recall": 0.0, "f1": -1.0}
    for t in ts:
        pred = (p >= t).astype(int)
        prec = precision_score(y, pred, zero_division=0)
        rec  = recall_score(y, pred, zero_division=0)
        f1   = f1_score(y, pred, zero_division=0)
        if precision_floor is None:
            if f1 > best["f1"]:
                best = {"t": float(t), "precision": prec, "recall": rec, "f1": f1}
        else:
            # only consider thresholds meeting the precision floor
            if prec >= precision_floor and f1 > best["f1"]:
                best = {"t": float(t), "precision": prec, "recall": rec, "f1": f1}
    # fallback if no threshold met the precision floor
    if precision_floor is not None and best["f1"] < 0:
        return pick_threshold(p, y, precision_floor=None, tmin=tmin, tmax=tmax, n=n)
    return best

calib_methods = ["sigmoid", "isotonic"]
strategies = {
    "f1_max": {"precision_floor": None},
    "f1_at_p70": {"precision_floor": 0.70}, 
}

results = []

# Evaluate each model + calibration method
for model_name, base_model in best_models.items():
    for method in calib_methods:
        print(f"\n=== {model_name} | calibration={method} ===")
        
        # Create calibrated wrapper
        cal = CalibratedClassifierCV(base_model, cv=5, method=method)
        cal.fit(X_subtr, y_subtr)

        # Get validation probabilities
        if hasattr(cal, "predict_proba"):
            p_val = cal.predict_proba(X_val)[:, 1]
        else:
            scores = cal.decision_function(X_val)
            smin, smax = scores.min(), scores.max()
            p_val = (scores - smin) / (smax - smin + 1e-9)

        # Strategy 1: pure F1 max
        s1 = pick_threshold(p_val, y_val, **strategies["f1_max"])
        # Strategy 2: F1 best at precision >= 0.70 (change floor above)
        s2 = pick_threshold(p_val, y_val, **strategies["f1_at_p70"])

        # Common ranking metrics (threshold-free)
        ap  = average_precision_score(y_val, p_val)
        roc = roc_auc_score(y_val, p_val)

        # Record both strategies
        results.append({
            "model": model_name,
            "calibration": method,
            "strategy": "f1_max",
            "best_threshold": s1["t"],
            "precision": s1["precision"],
            "recall": s1["recall"],
            "f1": s1["f1"],
            "AP": ap,
            "ROC_AUC": roc
        })
        results.append({
            "model": model_name,
            "calibration": method,
            "strategy": "f1_at_p70",
            "best_threshold": s2["t"],
            "precision": s2["precision"],
            "recall": s2["recall"],
            "f1": s2["f1"],
            "AP": ap,
            "ROC_AUC": roc
        })

# Build comparison table
calib_threshold_df = pd.DataFrame(results).sort_values(
    by=["model", "calibration", "strategy"]
).reset_index(drop=True)

print("\n[CALIBRATION / THRESHOLD COMPARISON]")
display(calib_threshold_df)

summary = (
    calib_threshold_df
    .sort_values(["model", "f1"], ascending=[True, False])
    .groupby("model")
    .head(2)  
    .reset_index(drop=True)
)
print("\n[TOP CHOICES PER MODEL]")
display(summary)

# Save to run dir
calib_threshold_df.to_csv(run_dir / "calibration_threshold_grid.csv", index=False)
summary.to_csv(run_dir / "calibration_threshold_summary.csv", index=False)
print(f"[INFO] Saved: {run_dir/'calibration_threshold_grid.csv'} and summary")


=== LogisticRegression | calibration=sigmoid ===

=== LogisticRegression | calibration=isotonic ===

=== LinearSVC | calibration=sigmoid ===

=== LinearSVC | calibration=isotonic ===

=== RandomForest | calibration=sigmoid ===

=== RandomForest | calibration=isotonic ===

=== GradientBoosting | calibration=sigmoid ===

=== GradientBoosting | calibration=isotonic ===

=== GaussianNB | calibration=sigmoid ===

=== GaussianNB | calibration=isotonic ===

[CALIBRATION / THRESHOLD COMPARISON]


,model,calibration,strategy,best_threshold,precision,recall,f1,AP,ROC_AUC
0,GaussianNB,isotonic,f1_at_p70,0.625,0.827586,0.827586,0.827586,0.725037,0.670259
1,GaussianNB,isotonic,f1_max,0.625,0.827586,0.827586,0.827586,0.725037,0.670259
2,GaussianNB,sigmoid,f1_at_p70,0.630,0.702703,0.896552,0.787879,0.780940,0.653017
3,GaussianNB,sigmoid,f1_max,0.570,0.659091,1.000000,0.794521,0.780940,0.653017
4,GradientBoosting,isotonic,f1_at_p70,0.595,0.705882,0.827586,0.761905,0.736661,0.628233
5,GradientBoosting,isotonic,f1_max,0.465,0.690476,1.000000,0.816901,0.736661,0.628233
6,GradientBoosting,sigmoid,f1_at_p70,0.645,0.733333,0.379310,0.500000,0.662763,0.433190
7,GradientBoosting,sigmoid,f1_max,0.050,0.644444,1.000000,0.783784,0.662763,0.433190
8,LinearSVC,isotonic,f1_at_p70,0.540,0.700000,0.965517,0.811594,0.699354,0.628233
9,LinearSVC,isotonic,f1_max,0.505,0.690476,1.000000,0.816901,0.699354,0.628233



[TOP CHOICES PER MODEL]


,model,calibration,strategy,best_threshold,precision,recall,f1,AP,ROC_AUC
0,GaussianNB,isotonic,f1_at_p70,0.625,0.827586,0.827586,0.827586,0.725037,0.670259
1,GaussianNB,isotonic,f1_max,0.625,0.827586,0.827586,0.827586,0.725037,0.670259
2,GradientBoosting,isotonic,f1_max,0.465,0.690476,1.000000,0.816901,0.736661,0.628233
3,GradientBoosting,sigmoid,f1_max,0.050,0.644444,1.000000,0.783784,0.662763,0.433190
4,LinearSVC,sigmoid,f1_at_p70,0.590,0.717949,0.965517,0.823529,0.699489,0.625000
5,LinearSVC,sigmoid,f1_max,0.590,0.717949,0.965517,0.823529,0.699489,0.625000
6,LogisticRegression,isotonic,f1_at_p70,0.435,0.700000,0.965517,0.811594,0.634578,0.515086
7,LogisticRegression,isotonic,f1_max,0.435,0.700000,0.965517,0.811594,0.634578,0.515086
8,RandomForest,isotonic,f1_at_p70,0.515,0.700000,0.965517,0.811594,0.678611,0.606681
9,RandomForest,isotonic,f1_max,0.515,0.700000,0.965517,0.811594,0.678611,0.606681


[INFO] Saved: /Users/kwill55/RVL/personalized-sports-optimization/experiments/summary_stats/2025-08-21_20-57-18_run-001_baseline/calibration_threshold_grid.csv and summary


In [140]:
# =================================================
# Step 4: Full Training & Testing (Evaluation)
# =================================================

# Use the train already built; keep test untouched
X_train_full = X_train
y_train_full = y_train

final_dir = run_dir / "final_eval"
final_dir.mkdir(parents=True, exist_ok=True)
print(f"[INFO] Saving final model evaluations to {final_dir}")

# Models to evaluate
chosen_models = ["LogisticRegression", "RandomForest", "GaussianNB", "LinearSVC", "GradientBoosting" ]  

# sanity checks
if 'best_models' not in globals():
    raise RuntimeError("best_models not found (run the hyperparameter tuning section first).")
if 'calib_threshold_df' not in globals():
    raise RuntimeError("calib_threshold_df not found (run the calibration/threshold section first).")

# Build: best calibration+threshold per model from calib_threshold_df
cfg_by_model = {}
for m in chosen_models:
    sub = calib_threshold_df[calib_threshold_df["model"] == m]
    if sub.empty:
        print(f"[WARN] No calibration rows for {m}; skipping.")
        continue
    # choose the row with highest F1 on your validation set
    row = sub.sort_values("f1", ascending=False).iloc[0]
    cfg_by_model[m] = {
        "calibration": row["calibration"],
        "threshold": float(row["best_threshold"]),
        "strategy": row["strategy"],
        "val_F1": float(row["f1"]),
        "val_precision": float(row["precision"]),
        "val_recall": float(row["recall"]),
        "val_AP": float(row["AP"]),
        "val_ROC_AUC": float(row["ROC_AUC"]),
    }

final_results = []
saved_paths = []

for m in chosen_models:
    if m not in cfg_by_model:
        continue
    if m not in best_models:
        print(f"[WARN] No tuned pipeline in best_models for {m}; skipping.")
        continue

    base = best_models[m]                                    
    method = cfg_by_model[m]["calibration"]
    thr    = cfg_by_model[m]["threshold"]

    print(f"\n=== {m}: train full (calibration={method}, thr={thr:.3f}) ===")
    cal = CalibratedClassifierCV(base, cv=5, method=method)  
    cal.fit(X_train_full, y_train_full)

    # predict on test
    p_test = cal.predict_proba(X_test)[:, 1]
    y_hat  = (p_test >= thr).astype(int)

    # metrics
    prec = precision_score(y_test, y_hat, zero_division=0)
    rec  = recall_score(y_test, y_hat, zero_division=0)
    f1   = f1_score(y_test, y_hat, zero_division=0)
    ap   = average_precision_score(y_test, p_test)
    roc  = roc_auc_score(y_test, p_test)
    bri  = brier_score_loss(y_test, p_test)
    cm   = confusion_matrix(y_test, y_hat)

    print(f"[TEST] {m}  P={prec:.3f} R={rec:.3f} F1={f1:.3f} AP={ap:.3f} ROC={roc:.3f} Brier={bri:.3f}")
    print(pd.DataFrame(cm, index=["true 0","true 1"], columns=["pred 0","pred 1"]))

    # save model + plots
    model_path = final_dir / f"{m}_cal-{method}.joblib"
    dump(cal, model_path)
    saved_paths.append(model_path.as_posix())

    fig_dir = final_dir / f"{m}_figs"
    save_all_eval_figures(
        outdir=fig_dir,
        y_true=y_test,
        y_pred=y_hat,
        y_proba=p_test,
        class_names=["neg","pos"]  # or ["miss","make"]
    )

    final_results.append({
        "model": m,
        "calibration": method,
        "threshold": thr,
        "val_metrics": {
            "F1": cfg_by_model[m]["val_F1"],
            "precision": cfg_by_model[m]["val_precision"],
            "recall": cfg_by_model[m]["val_recall"],
            "AP": cfg_by_model[m]["val_AP"],
            "ROC_AUC": cfg_by_model[m]["val_ROC_AUC"],
            "strategy": cfg_by_model[m]["strategy"],
        },
        "test_metrics": {
            "F1": float(f1),
            "precision": float(prec),
            "recall": float(rec),
            "AP": float(ap),
            "ROC_AUC": float(roc),
            "Brier": float(bri),
            "confusion_matrix": cm.tolist(),
        }
    })

# save summary
if final_results:
    final_df = pd.DataFrame([{
        "model": r["model"],
        "calibration": r["calibration"],
        "threshold": r["threshold"],
        "val_F1": r["val_metrics"]["F1"],
        "test_F1": r["test_metrics"]["F1"],
        "test_precision": r["test_metrics"]["precision"],
        "test_recall": r["test_metrics"]["recall"],
        "test_AP": r["test_metrics"]["AP"],
        "test_ROC_AUC": r["test_metrics"]["ROC_AUC"],
        "test_Brier": r["test_metrics"]["Brier"],
    } for r in final_results]).sort_values("test_F1", ascending=False).reset_index(drop=True)

    display(final_df)
    (final_dir / "final_results.json").write_text(json.dumps(final_results, indent=2))
    final_df.to_csv(final_dir / "final_results.csv", index=False)
    print(f"\n[INFO] Saved models:\n  - " + "\n  - ".join(saved_paths))
    print(f"[INFO] Results:\n  - {final_dir/'final_results.json'}\n  - {final_dir/'final_results.csv'}")
else:
    print("[WARN] Nothing trained—check chosen_models & that Step 5 produced calib_threshold_df rows.")


[INFO] Saving final model evaluations to /Users/kwill55/RVL/personalized-sports-optimization/experiments/summary_stats/2025-08-21_20-57-18_run-001_baseline/final_eval

=== LogisticRegression: train full (calibration=isotonic, thr=0.435) ===
[TEST] LogisticRegression  P=0.694 R=0.944 F1=0.800 AP=0.720 ROC=0.667 Brier=0.210
        pred 0  pred 1
true 0       6      15
true 1       2      34


/opt/anaconda3/envs/RVL/lib/python3.12/site-packages/sklearn/metrics/_plot/roc_curve.py:189: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  self.ax_.legend(loc="lower right")



=== RandomForest: train full (calibration=isotonic, thr=0.515) ===
[TEST] RandomForest  P=0.723 R=0.944 F1=0.819 AP=0.805 ROC=0.760 Brier=0.193
        pred 0  pred 1
true 0       8      13
true 1       2      34


/opt/anaconda3/envs/RVL/lib/python3.12/site-packages/sklearn/metrics/_plot/roc_curve.py:189: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  self.ax_.legend(loc="lower right")



=== GaussianNB: train full (calibration=isotonic, thr=0.625) ===
[TEST] GaussianNB  P=0.682 R=0.833 F1=0.750 AP=0.745 ROC=0.614 Brier=0.238
        pred 0  pred 1
true 0       7      14
true 1       6      30


...

/opt/anaconda3/envs/RVL/lib/python3.12/site-packages/sklearn/metrics/_plot/roc_curve.py:189: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  self.ax_.legend(loc="lower right")



=== GradientBoosting: train full (calibration=isotonic, thr=0.465) ===
[TEST] GradientBoosting  P=0.642 R=0.944 F1=0.764 AP=0.778 ROC=0.747 Brier=0.207
        pred 0  pred 1
true 0       2      19
true 1       2      34


/opt/anaconda3/envs/RVL/lib/python3.12/site-packages/sklearn/metrics/_plot/roc_curve.py:189: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  self.ax_.legend(loc="lower right")


,model,calibration,threshold,val_F1,test_F1,test_precision,test_recall,test_AP,test_ROC_AUC,test_Brier
0,RandomForest,isotonic,0.515,0.811594,0.819277,0.723404,0.944444,0.804978,0.759921,0.193302
1,LogisticRegression,isotonic,0.435,0.811594,0.800000,0.693878,0.944444,0.719732,0.666667,0.210359
2,LinearSVC,sigmoid,0.590,0.823529,0.780488,0.695652,0.888889,0.764435,0.664021,0.214405
3,GradientBoosting,isotonic,0.465,0.816901,0.764045,0.641509,0.944444,0.777701,0.746693,0.207215
4,GaussianNB,isotonic,0.625,0.827586,0.750000,0.681818,0.833333,0.744855,0.614418,0.238315



[INFO] Saved models:
  - /Users/kwill55/RVL/personalized-sports-optimization/experiments/summary_stats/2025-08-21_20-57-18_run-001_baseline/final_eval/LogisticRegression_cal-isotonic.joblib
  - /Users/kwill55/RVL/personalized-sports-optimization/experiments/summary_stats/2025-08-21_20-57-18_run-001_baseline/final_eval/RandomForest_cal-isotonic.joblib
  - /Users/kwill55/RVL/personalized-sports-optimization/experiments/summary_stats/2025-08-21_20-57-18_run-001_baseline/final_eval/GaussianNB_cal-isotonic.joblib
  - /Users/kwill55/RVL/personalized-sports-optimization/experiments/summary_stats/2025-08-21_20-57-18_run-001_baseline/final_eval/LinearSVC_cal-sigmoid.joblib
  - /Users/kwill55/RVL/personalized-sports-optimization/experiments/summary_stats/2025-08-21_20-57-18_run-001_baseline/final_eval/GradientBoosting_cal-isotonic.joblib
[INFO] Results:
  - /Users/kwill55/RVL/personalized-sports-optimization/experiments/summary_stats/2025-08-21_20-57-18_run-001_baseline/final_eval/final_results

In [141]:
# ============================================================
# STEP 5: Diagnostics of final models 
# ============================================================

# --------------------------
# Paths & inputs
# --------------------------
final_dir = run_dir / "final_eval"
results_json = final_dir / "final_results.json"
if not results_json.exists():
    raise FileNotFoundError(f"Expected {results_json} from Step 4 at: {results_json}")

with results_json.open() as f:
    final_results = json.load(f)

X_train_full = X_train
y_train_full = y_train

diag_root = run_dir / "diagnostics_final"
diag_root.mkdir(parents=True, exist_ok=True)

if 'X_val' in globals() and 'y_val' in globals():
    use_X_perm, use_y_perm, perm_where = X_val, y_val, "val"
else:
    use_X_perm, use_y_perm, perm_where = X_test, y_test, "test"
print(f"[INFO] Permutation importance will run on: {perm_where} set")

# --------------------------
# Helpers for importances
# --------------------------
def _clf_and_scaler_from_pipeline(pipeline):
    """Return (clf, scaler_or_None). Assumes pipeline steps named 'scaler' and 'clf'."""
    if hasattr(pipeline, "named_steps"):
        scaler = pipeline.named_steps.get("scaler", None)
        clf    = pipeline.named_steps.get("clf", None)
        clf    = clf if clf is not None else pipeline
    else:
        scaler, clf = None, pipeline
    return clf, scaler

def _linear_coef_df(pipeline, feature_names):
    """Coefficients in ORIGINAL feature units if StandardScaler present."""
    clf, scaler = _clf_and_scaler_from_pipeline(pipeline)
    if not hasattr(clf, "coef_"):
        return None
    coef = np.ravel(clf.coef_)
    if scaler is not None and hasattr(scaler, "scale_"):
        scale = np.where(np.asarray(scaler.scale_) == 0, 1.0, np.asarray(scaler.scale_))
        coef = coef / scale
    df = pd.DataFrame({"feature": feature_names, "coef": coef})
    df["abs_coef"] = df["coef"].abs()
    return df.sort_values("abs_coef", ascending=False).reset_index(drop=True)

def _tree_importance_df(pipeline, feature_names):
    """Impurity-based importances for RF/GB."""
    clf, _ = _clf_and_scaler_from_pipeline(pipeline)
    if not hasattr(clf, "feature_importances_"):
        return None
    imp = np.asarray(clf.feature_importances_, dtype=float)
    df = pd.DataFrame({"feature": feature_names, "impurity_importance": imp})
    return df.sort_values("impurity_importance", ascending=False).reset_index(drop=True)

def _permutation_importance_df(
    estimator, X_perm, y_perm, feature_names,
    mode="roc_auc",  # "roc_auc" | "average_precision" | "f1_at_threshold"
    threshold=None,
    n_repeats=30
):
    """
    mode:
      - "roc_auc": threshold-free, uses predict_proba
      - "average_precision": threshold-free PR AUC
      - "f1_at_threshold": applies 'threshold' to predict_proba[:,1] before F1
    """
    if mode in ("roc_auc", "average_precision"):
        scoring = mode
    elif mode == "f1_at_threshold":
        if threshold is None:
            raise ValueError("threshold must be provided for f1_at_threshold")
        def _score(est, X, y):
            p = est.predict_proba(X)[:, 1]
            yhat = (p >= threshold).astype(int)
            return _f1_for_perm(y, yhat, zero_division=0)
        scoring = _score
    else:
        raise ValueError(f"Unknown mode: {mode}")

    res = permutation_importance(
        estimator, X_perm, y_perm,
        scoring=scoring, n_repeats=n_repeats, random_state=42, n_jobs=-1
    )
    df = pd.DataFrame({
        "feature": feature_names,
        "perm_importance_mean": res.importances_mean,
        "perm_importance_std":  res.importances_std
    })
    return df.sort_values("perm_importance_mean", ascending=False).reset_index(drop=True)

# --------------------------
# Loop over final models
# --------------------------
summary_rows = []
feat_names = list(X_train_full.columns)

for entry in final_results:
    m        = entry["model"]
    method   = entry["calibration"]
    thr      = float(entry["threshold"])
    model_path = final_dir / f"{m}_cal-{method}.joblib"

    if not model_path.exists():
        print(f"[WARN] Missing saved model for {m} at {model_path}; skipping.")
        continue

    print(f"\n=== {m} | load final model | calibration={method} | threshold={thr:.3f} ===")
    cal = load(model_path)  # FINAL trained + calibrated model (CalibratedClassifierCV)

    # --- Performance on TEST (no refit) ---
    p_test = cal.predict_proba(X_test)[:, 1]
    y_hat  = (p_test >= thr).astype(int)

    prec = precision_score(y_test, y_hat, zero_division=0)
    rec  = recall_score(y_test, y_hat, zero_division=0)
    f1   = f1_score(y_test, y_hat, zero_division=0)
    ap   = average_precision_score(y_test, p_test)
    roc  = roc_auc_score(y_test, p_test)
    bri  = brier_score_loss(y_test, p_test)
    cm   = confusion_matrix(y_test, y_hat)
    print(f"[TEST] P={prec:.3f} R={rec:.3f} F1={f1:.3f} AP={ap:.3f} ROC={roc:.3f} Brier={bri:.3f}")

    # --- Save plots ---
    outdir = (diag_root / m)
    figdir = outdir / "figs"
    outdir.mkdir(parents=True, exist_ok=True)
    save_all_eval_figures(figdir, y_test, y_hat, p_test, class_names=["neg","pos"])

    # --- Save predictions & errors ---
    errs = X_test.copy()
    errs["y_true"] = y_test.values
    errs["y_prob"] = p_test
    errs["y_pred"] = y_hat
    errs["error_type"] = np.where((errs["y_true"] == 1) & (errs["y_pred"] == 0), "FN",
                          np.where((errs["y_true"] == 0) & (errs["y_pred"] == 1), "FP", "OK"))
    try:
        errs["session"] = session_test.values
    except Exception:
        pass
    errs.to_csv(outdir / "predictions_with_errors.csv", index=False)
    errs[errs["error_type"].isin(["FP","FN"])].to_csv(outdir / "misclassified_only.csv", index=False)

    # --- Permutation importances on final calibrated model ---
    # 1) Threshold-free (ROC AUC)
    perm_auc = _permutation_importance_df(
        cal, use_X_perm, use_y_perm, feat_names,
        mode="roc_auc", n_repeats=30
    )
    perm_auc.to_csv(outdir / f"importance_perm_rocauc_on_{perm_where}.csv", index=False)

    # 2) F1 at actual operating threshold
    perm_f1thr = _permutation_importance_df(
        cal, use_X_perm, use_y_perm, feat_names,
        mode="f1_at_threshold", threshold=thr, n_repeats=30
    )
    perm_f1thr.to_csv(outdir / f"importance_perm_f1_thr{thr:.3f}_on_{perm_where}.csv", index=False)

    # --- Model-native importances ---
    if 'best_models' in globals() and m in best_models:
        base = best_models[m]
        base_fitted = base.fit(X_train_full, y_train_full)

        coef_df = _linear_coef_df(base_fitted, feat_names)
        if coef_df is not None:
            coef_df.to_csv(outdir / "importance_coefficients.csv", index=False)

        tree_df = _tree_importance_df(base_fitted, feat_names)
        if tree_df is not None:
            tree_df.to_csv(outdir / "importance_impurity.csv", index=False)

    # --- Config snapshot (from Step 4 entry + new test metrics) ---
    tn, fp, fn, tp = cm.ravel()
    meta = {
        "model": m,
        "calibration": method,
        "threshold": thr,
        "val_metrics_from_step6": entry.get("val_metrics", {}),
        "test_metrics": {
            "precision": float(prec),
            "recall": float(rec),
            "f1": float(f1),
            "AP": float(ap),
            "ROC_AUC": float(roc),
            "Brier": float(bri),
            "confusion_matrix": [int(tn), int(fp), int(fn), int(tp)]
        }
    }
    (outdir / "config_metrics.json").write_text(json.dumps(meta, indent=2))

    # summary row
    summary_rows.append({
        "model": m, "calibration": method, "threshold": thr,
        "precision": float(prec), "recall": float(rec), "f1": float(f1),
        "AP": float(ap), "ROC_AUC": float(roc), "Brier": float(bri),
        "perm_top_feature_rocauc": (perm_auc.iloc[0]["feature"] if len(perm_auc) else None),
        "perm_top_feature_f1thr": (perm_f1thr.iloc[0]["feature"] if len(perm_f1thr) else None),
    })

# Roll-up table
if summary_rows:
    summary_df = pd.DataFrame(summary_rows).sort_values("f1", ascending=False).reset_index(drop=True)
    display(summary_df)
    summary_df.to_csv(diag_root / "summary_across_models.csv", index=False)
    print(f"\n[INFO] Diagnostics written under: {diag_root}")
else:
    print("[WARN] No models diagnosed — check Step 6 outputs in", final_dir)


[INFO] Permutation importance will run on: val set

=== LogisticRegression | load final model | calibration=isotonic | threshold=0.435 ===
[TEST] P=0.694 R=0.944 F1=0.800 AP=0.720 ROC=0.667 Brier=0.210


/opt/anaconda3/envs/RVL/lib/python3.12/site-packages/sklearn/metrics/_plot/roc_curve.py:189: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  self.ax_.legend(loc="lower right")



=== RandomForest | load final model | calibration=isotonic | threshold=0.515 ===
[TEST] P=0.723 R=0.944 F1=0.819 AP=0.805 ROC=0.760 Brier=0.193


/opt/anaconda3/envs/RVL/lib/python3.12/site-packages/sklearn/metrics/_plot/roc_curve.py:189: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  self.ax_.legend(loc="lower right")



=== GaussianNB | load final model | calibration=isotonic | threshold=0.625 ===
[TEST] P=0.682 R=0.833 F1=0.750 AP=0.745 ROC=0.614 Brier=0.238


...

/opt/anaconda3/envs/RVL/lib/python3.12/site-packages/sklearn/metrics/_plot/roc_curve.py:189: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  self.ax_.legend(loc="lower right")



=== GradientBoosting | load final model | calibration=isotonic | threshold=0.465 ===
[TEST] P=0.642 R=0.944 F1=0.764 AP=0.778 ROC=0.747 Brier=0.207


/opt/anaconda3/envs/RVL/lib/python3.12/site-packages/sklearn/metrics/_plot/roc_curve.py:189: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  self.ax_.legend(loc="lower right")


,model,calibration,threshold,precision,recall,f1,AP,ROC_AUC,Brier,perm_top_feature_rocauc,perm_top_feature_f1thr
0,RandomForest,isotonic,0.515,0.723404,0.944444,0.819277,0.804978,0.759921,0.193302,elbow_flex_l_follow_total_change,shoulder_flex_l_windup_min
1,LogisticRegression,isotonic,0.435,0.693878,0.944444,0.800000,0.719732,0.666667,0.210359,ankle_flex_r_follow_std,ankle_flex_r_follow_std
2,LinearSVC,sigmoid,0.590,0.695652,0.888889,0.780488,0.764435,0.664021,0.214405,elbow_flex_l_follow_range,ankle_flex_r_follow_std
3,GradientBoosting,isotonic,0.465,0.641509,0.944444,0.764045,0.777701,0.746693,0.207215,elbow_flex_l_follow_total_change,shoulder_flex_l_windup_min
4,GaussianNB,isotonic,0.625,0.681818,0.833333,0.750000,0.744855,0.614418,0.238315,hip_flex_r_windup_total_change,hip_flex_r_follow_std



[INFO] Diagnostics written under: /Users/kwill55/RVL/personalized-sports-optimization/experiments/summary_stats/2025-08-21_20-57-18_run-001_baseline/diagnostics_final


In [132]:
# ===== Step 6: Feature Pruning from Permutation Importance =====

#sel_ang_and_static
EXPERIMENT = "2025-08-21_20-02-28_run-001_baseline" # <-- adjust path

# --- Config ---
EXPERIMENT_DIR = Path("../experiments") / EXPERIMENT 
IMPORTANCE_DIR = EXPERIMENT_DIR / "diagnostics_final"
OUTPUT_LIST    = EXPERIMENT_DIR / "reduced_features.csv"
OUTPUT_MERGED  = EXPERIMENT_DIR / "feature_importance_merged.csv"
THRESHOLD      = 0.01  # keep if mean_importance >= THRESHOLD after clipping negatives to 0

# Recursively find permutation importance CSVs under each model dir
files = list(IMPORTANCE_DIR.glob("**/importance_perm_*_on_val.csv"))
if not files:
    raise FileNotFoundError(f"No permutation CSVs found under {IMPORTANCE_DIR} (**/importance_perm_*_on_val.csv)")

# ---- Load all to long-form ----
rows = []
for f in files:
    # model name is the immediate parent directory (e.g., LinearSVC)
    model = f.parent.name
    # metric is everything between 'importance_perm_' and '_on_val'
    m = re.search(r"importance_perm_(.+)_on_val\.csv$", f.name)
    metric = m.group(1) if m else "unknown"

    df = pd.read_csv(f)  # expects: feature, perm_importance_mean, perm_importance_std
    if {"feature","perm_importance_mean","perm_importance_std"} - set(df.columns):
        raise ValueError(f"Unexpected columns in {f}: {df.columns.tolist()}")

    tmp = df[["feature","perm_importance_mean","perm_importance_std"]].copy()
    tmp["metric"] = metric
    tmp["model"]  = model
    rows.append(tmp)

long_df = pd.concat(rows, ignore_index=True)
long_df["perm_importance_mean"] = long_df["perm_importance_mean"].clip(lower=0)

# ---- Aggregate across models for each (feature, metric) ----
agg_df = (
    long_df
    .groupby(["feature","metric"], as_index=False)
    .agg(mean_over_models=("perm_importance_mean","mean"),
         std_over_models=("perm_importance_mean","std"))
)

# ---- Pivot to wide: one row per feature, columns = metrics (averaged over models) ----
wide = agg_df.pivot_table(index="feature", columns="metric", values="mean_over_models", aggfunc="first").fillna(0.0)
# flatten column index and prefix with 'mean_'
wide.columns = [f"mean_{c}" for c in wide.columns]
wide = wide.reset_index()

# ---- Overall mean importance across all metrics ----
mean_cols = [c for c in wide.columns if c.startswith("mean_")]
wide["mean_importance"] = wide[mean_cols].mean(axis=1)

# ---- Filter by threshold and save ----
reduced = wide[wide["mean_importance"] >= THRESHOLD].sort_values("mean_importance", ascending=False)
reduced["feature"].to_csv(OUTPUT_LIST, index=False)
wide.sort_values("mean_importance", ascending=False).to_csv(OUTPUT_MERGED, index=False)

print(f"[INFO] Files loaded: {len(files)}")
print(f"[INFO] Features kept: {len(reduced)} / {len(wide)} (threshold={THRESHOLD})")
print(f"[INFO] Saved reduced list -> {OUTPUT_LIST}")
print(f"[INFO] Saved merged table -> {OUTPUT_MERGED}")


[INFO] Files loaded: 10
[INFO] Features kept: 35 / 51 (threshold=0.01)
[INFO] Saved reduced list -> ../experiments/2025-08-21_20-02-28_run-001_baseline/reduced_features.csv
[INFO] Saved merged table -> ../experiments/2025-08-21_20-02-28_run-001_baseline/feature_importance_merged.csv
